## 1. Выгрузка данных

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv


In [2]:
# Загружаем переменные из файла .env 
load_dotenv() 
# Получаем адрес подключения к БД 
database_url = os.getenv("DATABASE_URL") 
    
# Создаём подключение к БД 
engine = create_engine(database_url) 

# Проверяем подключение 
with engine.connect(): 
    print("Подключение к БД успешно") 
    
# Выполняем SQL-запрос и получаем DataFrame 
query = '''
        SELECT *
        FROM raw_banks_data
    '''
df = pd.read_sql(query, engine)
print(df.head())

Подключение к БД успешно
   children  days_employed  dob_years education  education_id  \
0         1   -8437.673028         42    высшее             0   
1         1   -4024.803754         36   среднее             1   
2         0   -5623.422610         33   Среднее             1   
3         3   -4124.747207         32   среднее             1   
4         0  340266.072047         53   среднее             1   

      family_status  family_status_id gender income_type  debt   total_income  \
0   женат / замужем                 0      F   сотрудник     0  253875.639453   
1   женат / замужем                 0      F   сотрудник     0  112080.014102   
2   женат / замужем                 0      M   сотрудник     0  145885.952297   
3   женат / замужем                 0      M   сотрудник     0  267628.550329   
4  гражданский брак                 1      F   пенсионер     0  158616.077870   

                      purpose  
0               покупка жилья  
1     приобретение автомобиля  
2

## 2. Предобработка данных

In [3]:
df.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


In [5]:
df = df.rename(columns={"dob_years": "client_age"})

In [6]:
for column in df:
    print(column, df[column].unique())

children [ 1  0  3  2 -1  4 20  5]
days_employed [-8437.67302776 -4024.80375385 -5623.42261023 ... -2113.3468877
 -3112.4817052  -1984.50758853]
client_age [42 36 33 32 53 27 43 50 35 41 40 65 54 56 26 48 24 21 57 67 28 63 62 47
 34 68 25 31 30 20 49 37 45 61 64 44 52 46 23 38 39 51  0 59 29 60 55 58
 71 22 73 66 69 19 72 70 74 75]
education ['высшее' 'среднее' 'Среднее' 'СРЕДНЕЕ' 'ВЫСШЕЕ' 'неоконченное высшее'
 'начальное' 'Высшее' 'НЕОКОНЧЕННОЕ ВЫСШЕЕ' 'Неоконченное высшее'
 'НАЧАЛЬНОЕ' 'Начальное' 'Ученая степень' 'УЧЕНАЯ СТЕПЕНЬ'
 'ученая степень']
education_id [0 1 2 3 4]
family_status ['женат / замужем' 'гражданский брак' 'вдовец / вдова' 'в разводе'
 'Не женат / не замужем']
family_status_id [0 1 2 3 4]
gender ['F' 'M' 'XNA']
income_type ['сотрудник' 'пенсионер' 'компаньон' 'госслужащий' 'безработный'
 'предприниматель' 'студент' 'в декрете']
debt [0 1]
total_income [253875.6394526  112080.01410244 145885.95229686 ...  89672.56115303
 244093.05050043  82047.41889948]
purpose ['п

In [7]:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.lower()

In [8]:
df['gender'] = df['gender'].replace({'f': 'жен', 'm': 'муж'})
df.drop(df[df['gender'] == 'xna'].index,inplace=True)

In [9]:
df['days_employed'] = df['days_employed'].abs()

In [10]:
df['client_age'].describe()

count    21524.000000
mean        43.294276
std         12.574188
min          0.000000
25%         33.000000
50%         42.000000
75%         53.000000
max         75.000000
Name: client_age, dtype: float64

In [11]:
df['total_income'].describe()

count    1.935000e+04
mean     1.674204e+05
std      1.029739e+05
min      2.066726e+04
25%      1.030407e+05
50%      1.450117e+05
75%      2.034244e+05
max      2.265604e+06
Name: total_income, dtype: float64

In [12]:
df = df.loc[
    (df['children'] >= 0)&
    (df['children'] != 20)
    ]

df = df.loc[
    (df['client_age'] >= 18) 
    ]    

In [13]:
df['income_type'].value_counts()

income_type
сотрудник          10996
компаньон           5033
пенсионер           3819
госслужащий         1447
безработный            2
предприниматель        2
студент                1
в декрете              1
Name: count, dtype: int64

In [14]:
df['family_status'].value_counts()

family_status
женат / замужем          12254
гражданский брак          4138
не женат / не замужем     2783
в разводе                 1179
вдовец / вдова             947
Name: count, dtype: int64

In [15]:
df['education'].value_counts()

education
среднее                15073
высшее                  5202
неоконченное высшее      738
начальное                282
ученая степень             6
Name: count, dtype: int64

In [16]:
# удалим единичные категории,т.к. для статистических выводов они не показательны и могут создавать выбросы
df.drop(df[
            (df['income_type'] == 'безработный')|
            (df['income_type'] == 'студент')|
            (df['income_type'] == 'предприниматель')|
            (df['income_type'] == 'в декрете')
    ].index,inplace=True)

df.drop(df[df['education'] == 'ученая степень'].index,inplace=True)

In [17]:
# добавим столбец
df['years_employed'] = df['days_employed']/365
df['years_employed'].describe()

count    19138.000000
mean       183.500868
std        381.072430
min          0.066141
25%          2.542002
50%          6.017830
75%         15.206545
max       1100.699727
Name: years_employed, dtype: float64

In [18]:
df.loc[df['years_employed']>=100,'income_type'].unique()

array(['пенсионер'], dtype=object)

#### аномалии в данных: у 3410 клиентов трудовой стаж указан более 100 лет, что физически невозможно. Все эти клиенты - пенсионеры Заменим на NA. Далее для анализа будеи смотреть только трудовой стаж работающих клиентов

In [19]:
df.loc[df['years_employed'] > 100, 'years_employed'] = np.nan

In [20]:
# удалим пропуски
df = df.dropna(subset=['days_employed', 'total_income'])


In [21]:
def get_purpose_group(purpose):
    '''Группирует цель кредита в одну из основных категорий.'''
    
    if 'жиль' in purpose or 'недвижим' in purpose or 'строительств' in purpose or 'ремонт' in purpose:
        return 'недвижимость'
    elif 'автомобил' in purpose:
        return 'автомобиль'
    elif 'образован' in purpose:
        return 'образование'
    elif 'свадьб' in purpose:
        return 'свадьба'
    else:
        return 'другое'


df['purpose_group'] = df['purpose'].apply(get_purpose_group)

In [22]:
print(df[['purpose', 'purpose_group']].head(20))

                                   purpose purpose_group
0                            покупка жилья  недвижимость
1                  приобретение автомобиля    автомобиль
2                            покупка жилья  недвижимость
3               дополнительное образование   образование
4                          сыграть свадьбу       свадьба
5                            покупка жилья  недвижимость
6                        операции с жильем  недвижимость
7                              образование   образование
8                    на проведение свадьбы       свадьба
9                  покупка жилья для семьи  недвижимость
10                    покупка недвижимости  недвижимость
11       покупка коммерческой недвижимости  недвижимость
13                 приобретение автомобиля    автомобиль
14              покупка жилой недвижимости  недвижимость
15  строительство собственной недвижимости  недвижимость
16                            недвижимость  недвижимость
17              строительство н

In [23]:
df['days_employed'] = df['days_employed'].astype(int)
df['total_income'] = df['total_income'].astype(int)

In [24]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
Index: 19138 entries, 0 to 21524
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          19138 non-null  int64  
 1   days_employed     19138 non-null  int64  
 2   client_age        19138 non-null  int64  
 3   education         19138 non-null  object 
 4   education_id      19138 non-null  int64  
 5   family_status     19138 non-null  object 
 6   family_status_id  19138 non-null  int64  
 7   gender            19138 non-null  object 
 8   income_type       19138 non-null  object 
 9   debt              19138 non-null  int64  
 10  total_income      19138 non-null  int64  
 11  purpose           19138 non-null  object 
 12  years_employed    15728 non-null  float64
 13  purpose_group     19138 non-null  object 
dtypes: float64(1), int64(7), object(6)
memory usage: 2.2+ MB


,children,days_employed,client_age,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,years_employed,purpose_group
0,1,8437,42,высшее,0,женат / замужем,0,жен,сотрудник,0,253875,покупка жилья,23.116912,недвижимость
1,1,4024,36,среднее,1,женат / замужем,0,жен,сотрудник,0,112080,приобретение автомобиля,11.026860,автомобиль
2,0,5623,33,среднее,1,женат / замужем,0,муж,сотрудник,0,145885,покупка жилья,15.406637,недвижимость
3,3,4124,32,среднее,1,женат / замужем,0,муж,сотрудник,0,267628,дополнительное образование,11.300677,образование
4,0,340266,53,среднее,1,гражданский брак,1,жен,пенсионер,0,158616,сыграть свадьбу,NaN,свадьба


In [25]:
for column in df:
    print(column, df[column].unique())

children [1 0 3 2 4 5]
days_employed [  8437   4024   5623 ... 362161 373995 343937]
client_age [42 36 33 32 53 27 43 50 35 41 40 54 56 26 48 24 21 57 67 28 62 47 34 68
 25 31 30 20 49 37 45 63 61 64 44 46 23 38 39 51 65 59 29 52 60 55 58 71
 22 73 66 69 19 72 70 74 75]
education ['высшее' 'среднее' 'неоконченное высшее' 'начальное']
education_id [0 1 2 3]
family_status ['женат / замужем' 'гражданский брак' 'вдовец / вдова' 'в разводе'
 'не женат / не замужем']
family_status_id [0 1 2 3 4]
gender ['жен' 'муж']
income_type ['сотрудник' 'пенсионер' 'компаньон' 'госслужащий']
debt [0 1]
total_income [253875 112080 145885 ...  89672 244093  82047]
purpose ['покупка жилья' 'приобретение автомобиля' 'дополнительное образование'
 'сыграть свадьбу' 'операции с жильем' 'образование'
 'на проведение свадьбы' 'покупка жилья для семьи' 'покупка недвижимости'
 'покупка коммерческой недвижимости' 'покупка жилой недвижимости'
 'строительство собственной недвижимости' 'недвижимость'
 'строительство не

In [26]:
print(df.duplicated().sum())

0


In [27]:
print(df.isna().sum())

children               0
days_employed          0
client_age             0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income           0
purpose                0
years_employed      3410
purpose_group          0
dtype: int64


## 3. Очищенный датафрейм загружаем в БД

In [28]:
# Загружаем данные:     
try:
    df.to_sql('clean_banks_data', engine, if_exists='replace', index=False)
    print(f"Данные успешно загружены в БД ")
except Exception as e:
    print(f"Ошибка при загрузке в БД: {e}")
   
    

Данные успешно загружены в БД 
